## Lab 4 - Build the Audience, Then Buy It

Name: Korinnyi Zakhar
Group: KM-43

### Setup

In [21]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


### Task 1.0 Clean the survey file

In [22]:
# adults from viewer_attributes
age = viewer_attributes[viewer_attributes['attr_id'] == 300].copy()
age['age'] = pd.to_numeric(age['value'], errors='coerce')
adults = age[age['age'] >= 18]['respondent_id'].unique()

s = survey.copy()
valid_kw = survey_layout[survey_layout['RECTYPE'] == 'B']['KEYWORD']
yes_values = ['1', '1.0', 'y', 'true', 'yes']
rows = []

# header and footer lines
s['id'] = pd.to_numeric(s['RESPID'].str.strip(), errors='coerce')
rows.append(['Header or footer line inside the data', s['id'].isna().sum(), 'Dropped'])
s = s[s['id'].notna()].copy()
s['id'] = s['id'].astype(int)

# ids with spaces or decimals
rows.append(['Id with spaces or decimals', (s['RESPID'] != s['id'].astype(str)).sum(), 'Fixed'])

# keywords
s['kw'] = s['KEYWORD'].str.strip().str.upper()
rows.append(['Keyword in lower case or with spaces', (s['KEYWORD'] != s['kw']).sum(), 'Fixed'])
m = s['kw'].isin(valid_kw)
rows.append(['Keyword not in layout', (~m).sum(), 'Dropped'])
s = s[m].copy()

# values
s['val'] = s['VALUE'].str.strip().str.lower()
rows.append(['Yes written as 1.0, y, Y, TRUE, yes', ((s['val'] != '1') & s['val'].isin(yes_values)).sum(), 'Fixed'])
m = s['val'].isin(yes_values)
rows.append(['Answer is not a yes (0, N, no, empty)', (~m).sum(), 'Dropped'])
s = s[m].copy()

# not adults
m = s['id'].isin(adults)
rows.append(['Not an adult panel member', (~m).sum(), 'Dropped'])
lost = s[~m]
s = s[m].copy()

# duplicates
m = ~s.duplicated(['id', 'kw'])
rows.append(['Same yes answer twice', (~m).sum(), 'Dropped'])
s = s[m].copy()

# one row per adult
tab = pd.crosstab(s['id'], s['kw']).reindex(index=adults, columns=valid_kw, fill_value=0)
pair_cols = ['X710101', 'X710102', 'X710103']
brands = ['X710201', 'X710202', 'X710203', 'X710298', 'X710299']
two_pairs = (tab[pair_cols].sum(axis=1) > 1).sum()
extra = (tab[pair_cols].sum(axis=1) - 1).clip(lower=0).sum()
rows.append(['Two pair counts for one person', extra, 'Kept the higher one'])
clean_yes = len(s) - extra

people = pd.DataFrame(index=adults)
people['pairs'] = '0'
people.loc[tab['X710101'] == 1, 'pairs'] = '1'
people.loc[tab['X710102'] == 1, 'pairs'] = '2'
people.loc[tab['X710103'] == 1, 'pairs'] = '3+'
people['stride'] = tab['X710201'] == 1
people['kestrel'] = tab['X710202'] == 1
people['easywalk'] = tab['X710203'] == 1
people['buyer'] = (tab['X710100'] == 1) | (people['pairs'] != '0') | (tab[brands].sum(axis=1) > 0)

# my counts for the layout
mine = tab.sum()
mine['X710100'] = people['buyer'].sum()
mine['X710101'] = (people['pairs'] == '1').sum()
mine['X710102'] = (people['pairs'] == '2').sum()
mine['X710103'] = (people['pairs'] == '3+').sum()
check = survey_layout[survey_layout['RECTYPE'] == 'B'][['KEYWORD', 'SAMPLE']].copy()
check['your count'] = check['KEYWORD'].map(mine)
check = check[['KEYWORD', 'your count', 'SAMPLE']]

print("Survey data rows:", len(survey))
print("Clean yes answers:", clean_yes)
print("Adult panel members:", len(adults))
print("Non-buyers:", (~people['buyer']).sum())
print("People with two pair counts:", two_pairs)
# table: kind of bad record, rows, what you did
print(pd.DataFrame(rows, columns=['kind of bad record', 'rows', 'what you did']).to_string(index=False))
# table: KEYWORD, your count, layout SAMPLE
print(check.to_string(index=False))

Survey data rows: 18349
Clean yes answers: 15397
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
                   kind of bad record  rows        what you did
Header or footer line inside the data     3             Dropped
           Id with spaces or decimals  1455               Fixed
 Keyword in lower case or with spaces  1104               Fixed
                Keyword not in layout   738             Dropped
  Yes written as 1.0, y, Y, TRUE, yes  1142               Fixed
Answer is not a yes (0, N, no, empty)   742             Dropped
            Not an adult panel member   423             Dropped
                Same yes answer twice   933             Dropped
       Two pair counts for one person   113 Kept the higher one
KEYWORD  your count SAMPLE
X710100        4779   4779
X710101        1810   1810
X710102        1308   1308
X710103        1661   1661
X710201        1563   1563
X710202         354    354
X710203        1050   1050
X710298        2739

**Answer:** The vendor file broke its promises in many ways. There were 3 header or footer lines inside the data, 1,455 ids written with spaces, decimals or a leading zero, 1,104 keywords in lower case or with spaces, 738 keywords that are not in the layout, 1,142 yes answers written as 1.0, y, Y, TRUE or yes, 742 answers that were not a yes at all, 423 rows of people who are not adult panel members, 933 repeated yes answers and 113 people with two pair counts. I fixed what only looked different, dropped what was wrong, and for people with two pair counts I kept the higher one. After this every count matches the layout SAMPLE exactly. If I had used the file as delivered, Kestrel would show 516 rows against the true 354, which is about 46 percent too many. Almost 89 percent of adults would appear in the file against the 59.7 percent who really bought athletic shoes, so the share of buyers would be overstated by about 29 points.

### Task 1.1 Four definitions

In [23]:
dw = daily_weights[daily_weights['resp_type'].isin(['R', 'L'])]
adults_25_54 = age[(age['age'] >= 25) & (age['age'] <= 54)]['respondent_id'].unique()
shoe = people[people['buyer']].index
heavy = people[people['pairs'] == '3+'].index
kestrel = people[people['kestrel']].index

# average of 31 daily sums of weights
def size(ids):
    return dw[dw['respondent_id'].isin(ids)]['weight'].sum() / dw['broadcast_date'].nunique() / 1e6

print("Adults 18+ (M):", round(size(adults), 1))
print("Shoe buyers (M):", round(size(shoe), 1))
print("Heavy buyers, 3+ pairs (M):", round(size(heavy), 1))
print("Kestrel buyers (M):", round(size(kestrel), 1))
print("Adults 25-54 (M):", round(size(adults_25_54), 1))
print("Kestrel buyers in panel (n):", len(kestrel))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


### Task 1.2 - Who are they?

In [24]:
sex = viewer_attributes[viewer_attributes['attr_id'].isin([201, 202])].drop_duplicates('respondent_id')
sex = sex.set_index('respondent_id')['attr_id'].map({201: 'M', 202: 'W'})
ages = age.drop_duplicates('respondent_id').set_index('respondent_id')['age']

# adults in-tab with sex and age group
d = dw[dw['respondent_id'].isin(adults)].copy()
d['group'] = d['respondent_id'].map(sex) + ' ' + pd.cut(d['respondent_id'].map(ages), [17, 34, 54, 120], labels=['18-34', '35-54', '55+']).astype(str)

groups = ['M 18-34', 'M 35-54', 'M 55+', 'W 18-34', 'W 35-54', 'W 55+']
base = d.groupby('group')['weight'].sum()
base = base / base.sum()
audiences = {'Adults 18+': adults, 'Shoe buyers': shoe, 'Heavy buyers': heavy,
             'Kestrel buyers': kestrel, 'Adults 25-54': adults_25_54}

rows = []
for name, ids in audiences.items():
    w = d[d['respondent_id'].isin(ids)].groupby('group')['weight'].sum()
    share = (w / w.sum()).reindex(groups).fillna(0)
    for g in groups:
        rows.append([name, g, share[g], share[g] / base[g] * 100])
res_1_2 = pd.DataFrame(rows, columns=['audience', 'group', 'share', 'index'])

# table: audience x group, share and index vs adults
print(res_1_2.to_string(index=False, formatters={'share': '{:.3f}'.format, 'index': '{:.1f}'.format}))

      audience   group share index
    Adults 18+ M 18-34 0.144 100.0
    Adults 18+ M 35-54 0.160 100.0
    Adults 18+   M 55+ 0.185 100.0
    Adults 18+ W 18-34 0.144 100.0
    Adults 18+ W 35-54 0.160 100.0
    Adults 18+   W 55+ 0.208 100.0
   Shoe buyers M 18-34 0.140  97.4
   Shoe buyers M 35-54 0.158  99.2
   Shoe buyers   M 55+ 0.186 100.8
   Shoe buyers W 18-34 0.140  97.6
   Shoe buyers W 35-54 0.178 111.4
   Shoe buyers   W 55+ 0.197  94.6
  Heavy buyers M 18-34 0.111  77.1
  Heavy buyers M 35-54 0.138  86.8
  Heavy buyers   M 55+ 0.189 102.2
  Heavy buyers W 18-34 0.139  97.0
  Heavy buyers W 35-54 0.226 141.9
  Heavy buyers   W 55+ 0.196  94.0
Kestrel buyers M 18-34 0.118  82.1
Kestrel buyers M 35-54 0.189 118.6
Kestrel buyers   M 55+ 0.126  68.2
Kestrel buyers W 18-34 0.181 126.0
Kestrel buyers W 35-54 0.265 165.8
Kestrel buyers   W 55+ 0.121  58.0
  Adults 25-54 M 18-34 0.174 121.2
  Adults 25-54 M 35-54 0.326 204.1
  Adults 25-54   M 55+ 0.000   0.0
  Adults 25-54 W 18-

**Answer:** I would hand the planner the heavy buyers, people who bought three or more pairs. This is exactly what the marketing team asked for, and it is a large audience of 49.6 million people with 1,661 panelists behind it, so the ratings and forecasts built on it are stable. Kestrel buyers are only 10.0 million people and 354 panelists, so they are much noisier, and adults 25 to 54 would leave out the 55 plus group, which is nearly 40 percent of heavy buyers. Shoe buyers in general look almost like all adults because about 59 percent of adults bought athletic shoes and buying depends only a little on age and sex, so all their indexes stay between 95 and 111. Heavy buyers are the group that really differs, with indexes from 77 up to 142, and they lean to women 35 to 54. One thing a fused purchase attribute cannot tell you is whether the people who watch a particular program really buy more shoes than other people of the same age, sex and income, because the link to viewing goes only through these demographics.

### Task 2.1 Programs by audience

In [25]:
dwa = dw[dw['respondent_id'].isin(adults)].copy()
dwa['day'] = pd.to_datetime(dwa['broadcast_date'])
size_a = dwa.groupby('day')['weight'].sum()
size_h = dwa[dwa['respondent_id'].isin(heavy)].groupby('day')['weight'].sum()
size_k = dwa[dwa['respondent_id'].isin(kestrel)].groupby('day')['weight'].sum()

# october prime telecasts of the 8 programs
pl = pd.DataFrame([['NBC', 'NFL Football'], ['ABC', 'NFL Football'], ['ABC', 'Dancing with the Stars'],
                   ['CBS', 'NCIS'], ['FNC', 'Hannity'], ['HGTV', 'Unsellable Houses'],
                   ['FOOD', 'Halloween Wars'], ['FX', 'Grotesquerie']],
                  columns=['network_code', 'program_name'])
pt = programs.merge(pl, on=['network_code', 'program_name'])
pt = pt[(pt['daypart'] == 'Prime') & (pd.to_datetime(pt['broadcast_date']).dt.month == 10)]

# adult viewings on these networks
v = viewings[viewings['resp_type'].isin(['R', 'L']) & viewings['respondent_id'].isin(adults)
             & viewings['network_code'].isin(pl['network_code'])]

# rate card, times shifted to the broadcast day
has_prog = rate_card['program_name'].fillna('') != ''
rc_prog = rate_card[has_prog]
rc_time = rate_card[~has_prog].copy()
rc_time['start_min'] = rc_time['start_time'].apply(lambda t: int(t[:2]) * 60 + int(t[3:]) - 360)
rc_time['end_min'] = rc_time['end_time'].apply(lambda t: int(t[:2]) * 60 + int(t[3:]) - 360)

rows = []
for _, t in pt.iterrows():
    day = pd.to_datetime(t['broadcast_date'])
    dur = (t['end_ts'] - t['start_ts']).total_seconds() / 60
    w = dwa[dwa['day'] == day].set_index('respondent_id')['weight']

    # minutes watched inside the telecast, weighted
    x = v[(v['network_code'] == t['network_code']) & (v['start_ts'] < t['end_ts']) & (v['end_ts'] > t['start_ts'])].copy()
    x['mins'] = (x['end_ts'].clip(upper=t['end_ts']) - x['start_ts'].clip(lower=t['start_ts'])).dt.total_seconds() / 60
    x['w'] = x['respondent_id'].map(w)
    x = x.dropna(subset=['w'])
    x['wm'] = x['w'] * x['mins']
    xh = x[x['respondent_id'].isin(heavy)]
    xk = x[x['respondent_id'].isin(kestrel)]
    va = x['wm'].sum() / dur
    vh = xh['wm'].sum() / dur
    vk = xk['wm'].sum() / dur

    # price of the :30, program rows win
    p = rc_prog[(rc_prog['network_code'] == t['network_code']) & (rc_prog['program_name'] == t['program_name'])]
    if len(p) > 0:
        r = p
    else:
        b = t['start_ts'] - pd.Timedelta(hours=6)
        m = b.hour * 60 + b.minute
        r = rc_time[(rc_time['network_code'] == t['network_code']) & rc_time['weekdays'].apply(lambda l: b.dayofweek in l)
                    & (rc_time['start_min'] <= m) & (rc_time['end_min'] > m)]

    rows.append({'program': t['program_name'] + ' (' + t['network_code'] + ')', 'date': day.date(),
                 'v_adult': va, 'v_heavy': vh, 'v_kestrel': vk,
                 'r_adult': va / size_a[day] * 100, 'r_heavy': vh / size_h[day] * 100, 'r_kestrel': vk / size_k[day] * 100,
                 'n_heavy': xh['respondent_id'].nunique(), 'n_kestrel': xk['respondent_id'].nunique(),
                 'cost': r['cost_30s_usd'].iloc[0], 'rc_daypart': r['daypart'].iloc[0]})
tc = pd.DataFrame(rows)

g = tc.groupby('program')[['v_adult', 'v_heavy', 'r_adult', 'r_heavy', 'cost']].mean()
res_2_1 = pd.DataFrame({'adults rating': g['r_adult'], 'target rating': g['r_heavy'],
                        'target viewers (k)': g['v_heavy'] / 1000, 'composition': g['v_heavy'] / g['v_adult'],
                        'index': g['r_heavy'] / g['r_adult'] * 100, 'cost per :30': g['cost'],
                        'adults CPM': g['cost'] / (g['v_adult'] / 1000), 'target CPM': g['cost'] / (g['v_heavy'] / 1000)})
res_2_1 = res_2_1.reindex(['NFL Football (NBC)', 'NFL Football (ABC)', 'Dancing with the Stars (ABC)', 'NCIS (CBS)',
                           'Hannity (FNC)', 'Unsellable Houses (HGTV)', 'Halloween Wars (FOOD)', 'Grotesquerie (FX)'])
res_2_1.index.name = 'program'

# table: 8 programs, the columns of Task 2.1
print(res_2_1.to_string(formatters={'adults rating': '{:.2f}'.format, 'target rating': '{:.2f}'.format,
                                    'target viewers (k)': '{:.1f}'.format, 'composition': '{:.3f}'.format,
                                    'index': '{:.1f}'.format, 'cost per :30': '{:.0f}'.format,
                                    'adults CPM': '{:.2f}'.format, 'target CPM': '{:.2f}'.format}))
print("Target universe (M):", round(size(heavy), 1))

                             adults rating target rating target viewers (k) composition index cost per :30 adults CPM target CPM
program                                                                                                                         
NFL Football (NBC)                    5.82          5.79             2865.0       0.195  99.3       487200      33.12     170.05
NFL Football (ABC)                    8.93          8.36             4149.1       0.184  93.6       723600      32.09     174.40
Dancing with the Stars (ABC)          3.25          3.14             1564.2       0.191  96.8       158600      19.33     101.39
NCIS (CBS)                            2.21          2.43             1206.3       0.216 110.1       178800      32.09     148.22
Hannity (FNC)                         1.09          0.84              416.9       0.152  77.1        39900      14.50      95.71
Unsellable Houses (HGTV)              0.50          0.64              317.2       0.249 126.8    

### Task 2.2 Index, volume or price?

In [26]:
print("Highest index (program):", res_2_1['index'].idxmax())
print("Most target viewers (program):", res_2_1['target viewers (k)'].idxmax())
print("Lowest target CPM (program):", res_2_1['target CPM'].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): NFL Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** These are not all the same program. Unsellable Houses on HGTV has the highest index at 126.8 and also the lowest target CPM at about 41 dollars, so for this show the best concentration and the best price come together. But it gives only 317 thousand heavy buyers in an average minute, while Monday Night Football on ABC gives 4.1 million, more than thirteen times as many, at a target CPM of about 174 dollars. The two football games have an index close to 100, so their audience is about as full of heavy buyers as the population, but their size is what makes them valuable for volume. To a client who wants only the highest index I would say that the index shows how concentrated the heavy buyers are and not how many of them we can reach. Like the shampoo example in Lecture 5, a plan built only on the top index would put the whole million into a small show, buy many spots that reach the same few people and give a lot of repeated exposure and very little reach. A better plan uses the cheap and concentrated shows for efficiency and adds a large show for volume.

### Task 2.3 - Kestrel buyers

In [27]:
k = tc.groupby('program')[['r_adult', 'r_kestrel']].mean()
res_2_3 = pd.DataFrame({'heavy-buyer index': res_2_1['index'], 'Kestrel index': k['r_kestrel'] / k['r_adult'] * 100})

# table: program, heavy-buyer index, Kestrel index
print(res_2_3.to_string(formatters={'heavy-buyer index': '{:.1f}'.format, 'Kestrel index': '{:.1f}'.format}))

                             heavy-buyer index Kestrel index
program                                                     
Dancing with the Stars (ABC)              96.8          82.7
Grotesquerie (FX)                         95.9         129.6
Halloween Wars (FOOD)                    116.2          95.2
Hannity (FNC)                             77.1          42.6
NCIS (CBS)                               110.1          56.5
NFL Football (ABC)                        93.6          85.9
NFL Football (NBC)                        99.3          98.5
Unsellable Houses (HGTV)                 126.8          76.9


**Answer:** The Kestrel index jumps much more because there are far fewer Kestrel buyers in the panel, only 354 against 1,661 heavy buyers, and on small programs often only a few of them watch a given telecast, on Hannity usually one to three. With so few people one person more or less changes the rating a lot, so the Kestrel index goes from 42.6 on Hannity to 129.6 on Grotesquerie, while the heavy buyer index stays between 77 and 127. On the big games, where about 70 Kestrel buyers watch each telecast, the Kestrel index of 86 and 99 is close to the heavy buyer index of 94 and 99. I would trust the heavy buyer index, because it rests on about five times more people. Even this index is shaky on small shows like Hannity, where only about 16 heavy buyers watch a telecast. To check, I would count how many panelists stand behind each index and compute its standard error, which grows as the number of panelists falls. I would also split the telecasts of a program into two halves and compare the two indexes, and if they are close to each other the index can be trusted.

### Task 3.1 Weights

In [28]:
# edit rule and eastern time
a = acr_sessions[(pd.to_datetime(acr_sessions['end_utc']) - pd.to_datetime(acr_sessions['start_utc'])).dt.total_seconds() >= 60].copy()
a['start'] = pd.to_datetime(a['start_utc']).dt.tz_convert('US/Eastern').dt.tz_localize(None)
a['end'] = pd.to_datetime(a['end_utc']).dt.tz_convert('US/Eastern').dt.tz_localize(None)
a['day'] = (a['start'] - pd.Timedelta(hours=6)).dt.date

# tvs matched to exactly one household
tv = acr_sessions.drop_duplicates('tv_id')[['tv_id', 'hashed_ip']].copy()
n_hh = hh_attributes.groupby('hashed_ip')['household_id'].nunique()
tv['n_hh'] = tv['hashed_ip'].map(n_hh).fillna(0)
tv = tv[tv['n_hh'] == 1]
hh = hh_attributes.drop_duplicates('hashed_ip')[['hashed_ip', 'head_age_band', 'income_band']]
tv = tv.merge(hh, on='hashed_ip')
tv['head_age'] = tv['head_age_band'].map(lambda x: '65+' if x == '65+' else 'Under 65')

# in-tab matched tvs per day and cell weights
intab = a[['tv_id', 'day']].drop_duplicates().merge(tv, on='tv_id')
cnt = intab.groupby(['day', 'head_age', 'income_band']).size().rename('n').reset_index()
cnt = cnt.merge(universe, on=['head_age', 'income_band'])
cnt['weight'] = cnt['tv_households'] / cnt['n']
intab = intab.merge(cnt[['day', 'head_age', 'income_band', 'weight']], on=['day', 'head_age', 'income_band'])
sums = intab.groupby('day')['weight'].sum() / 1e6

# the eight reference telecasts
ref = reference_ratings.merge(programs[['telecast_id', 'network_code', 'program_name', 'start_ts', 'end_ts', 'broadcast_date']], on='telecast_id')
live = a[a['session_type'] == 'live']
rows = []
for _, r in ref.iterrows():
    d = intab[intab['day'] == pd.to_datetime(r['broadcast_date']).date()]
    s = live[live['network_code'] == r['network_code']]
    sec = (s['end'].clip(upper=r['end_ts']) - s['start'].clip(lower=r['start_ts'])).dt.total_seconds()
    watched = d[d['tv_id'].isin(s.loc[sec >= 60, 'tv_id'])]
    unw = len(watched) / len(d) * 100
    wtd = watched['weight'].sum() / d['weight'].sum() * 100
    rows.append([r['program_name'] + ' (' + r['network_code'] + ')', unw, wtd, r['hh_rating']])
res_3_1 = pd.DataFrame(rows, columns=['telecast', 'unweighted', 'weighted', 'reference'])

gap_u = (res_3_1['reference'] - res_3_1['unweighted']).abs().mean()
gap_w = (res_3_1['reference'] - res_3_1['weighted']).abs().mean()

print("TVs matched to one household:", len(tv))
print("Weights sum per day (M):", round(sums.mean(), 1))
# table: telecast, unweighted, weighted, reference
print(res_3_1.to_string(index=False, formatters={'unweighted': '{:.2f}'.format, 'weighted': '{:.2f}'.format, 'reference': '{:.2f}'.format}))
print("Mean gap unweighted (points):", round(gap_u, 2))
print("Mean gap weighted (points):", round(gap_w, 2))
print("Share of gap closed:", round((gap_u - gap_w) / gap_u, 3))

TVs matched to one household: 19607
Weights sum per day (M): 125.8
                                    telecast unweighted weighted reference
                        MLB Postseason (FOX)      17.03    17.76     25.86
                          NFL Football (NBC)      22.62    23.75     33.14
                                  NCIS (CBS)       5.50     5.80     10.56
                               Hannity (FNC)       1.04     1.35      2.30
                    Unsellable Houses (HGTV)       2.12     2.08      3.69
        Halloween Baking Championship (FOOD)       0.95     0.90      1.63
                           Grotesquerie (FX)       2.17     2.11      3.53
The Real Housewives of New York City (BRAVO)       0.71     0.64      1.23
Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


### Task 3.2 - What weighting cannot fix

**Answer:** Weighting moved the ratings only a little and in a clear pattern. The ratings of programs with older audiences went up, Hannity by 0.31 points, NCIS by 0.29, Sunday Night Football by 1.13 and the MLB postseason by 0.72. The ratings of the younger cable shows Unsellable Houses, Halloween Baking Championship, Grotesquerie and The Real Housewives went down slightly, by 0.04 to 0.07 points. This happens because the smart TVs sit too often in younger and richer homes. Homes with a head of household aged 65 or more are 19 percent of the footprint against 28 percent of the universe, and the poorest of them reach only 46 percent of their share, while homes with a younger head and an income of 75 thousand dollars or more are over represented by 17 to 38 percent. Weighting gives the missing older homes a bigger weight, so shows watched by older people gain and shows watched by younger people lose. Weighting closed only 7.5 percent of the mean gap, from 3.73 to 3.44 points, and after weighting the smart TVs still show only about 52 to 72 percent of the reference rating. What is left cannot be fixed by weights, because weights only correct who is in the sample and not what the TV can see. The feed has one TV per home out of about 2.6, it labels a large part of viewing as unknown, it misses homes that never opted in and TVs that could not be matched to one household, and inside a cell the TVs that agreed to share data may still differ from the ones that did not. Weighting by income barely moves these ratings because the cells that need the biggest correction are small, the poorest older homes are under 3 percent of the footprint, so even doubling their weight changes the total very little. The big gap is a loss of viewing that is spread over every cell in the same way, and a cell weight cannot bring it back.

### Task 4.1 October predicts October

In [29]:
nets = ['CBS', 'NBC', 'ABC', 'FOX', 'FNC']
p = programs[(programs['daypart'] == 'Prime') & programs['network_code'].isin(nets)].copy()
p['day'] = pd.to_datetime(p['broadcast_date'])
p = p[(p['day'] >= '2024-10-01') & (p['day'] <= '2024-10-28')]
p['program'] = p['program_name'] + ' (' + p['network_code'] + ')'

# daily weights and audience sizes
dwd = dw.copy()
dwd['day'] = pd.to_datetime(dwd['broadcast_date'])
w_day = {d: g.set_index('respondent_id')['weight'] for d, g in dwd.groupby('day')}
size_p2 = dwd.groupby('day')['weight'].sum()
size_hv = dwd[dwd['respondent_id'].isin(heavy)].groupby('day')['weight'].sum()

# person viewings by network
vn = {n: viewings[viewings['resp_type'].isin(['R', 'L']) & (viewings['network_code'] == n)] for n in nets}

rows = []
for _, t in p.iterrows():
    dur = (t['end_ts'] - t['start_ts']).total_seconds() / 60
    x = vn[t['network_code']]
    x = x[(x['start_ts'] < t['end_ts']) & (x['end_ts'] > t['start_ts'])].copy()
    mins = (x['end_ts'].clip(upper=t['end_ts']) - x['start_ts'].clip(lower=t['start_ts'])).dt.total_seconds() / 60
    x['wm'] = x['respondent_id'].map(w_day[t['day']]) * mins
    x = x.dropna(subset=['wm'])
    xh = x[x['respondent_id'].isin(heavy)]
    rows.append({'program': t['program'], 'date': t['day'],
                 'p2': x['wm'].sum() / dur / size_p2[t['day']] * 100,
                 'hv': xh['wm'].sum() / dur / size_hv[t['day']] * 100,
                 'n_p2': x['respondent_id'].nunique(), 'n_hv': xh['respondent_id'].nunique()})
tc4 = pd.DataFrame(rows)

# history Oct 1-21, forecast Oct 22-28
hist = tc4[tc4['date'] <= '2024-10-21']
fc = tc4[tc4['date'] >= '2024-10-22']
mh = hist.groupby('program')[['p2', 'hv']].mean()
fc = fc[fc['program'].isin(mh.index)].copy()
fc['f_p2'] = fc['program'].map(mh['p2'])
fc['f_hv'] = fc['program'].map(mh['hv'])

# skip telecasts with actual rating 0
a = fc[fc['p2'] > 0].copy()
b = fc[fc['hv'] > 0].copy()
a['err'] = (a['p2'] - a['f_p2']).abs() / a['p2']
b['err'] = (b['hv'] - b['f_hv']).abs() / b['hv']
worst = a.loc[a['err'].idxmax()]

print("Telecasts forecast (n):", len(fc))
print("MAPE, P2+ (%):", round(a['err'].mean() * 100, 1))
print("MAPE, heavy buyers (%):", round(b['err'].mean() * 100, 1))
print("Biggest miss, P2+ (program, date):", worst['program'] + ', ' + str(worst['date'].date()))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


### Task 4.2 Why it missed

In [30]:
top = a.nlargest(5, 'err')
res_4_2 = pd.DataFrame({'program': top['program'], 'date': top['date'].dt.date,
                        'forecast': top['f_p2'], 'actual': top['p2']})

# table: the five biggest misses (program, date, forecast, actual)
print(res_4_2.to_string(index=False, formatters={'forecast': '{:.2f}'.format, 'actual': '{:.2f}'.format}))

                   program       date forecast actual
          Krapopolis (FOX) 2024-10-27     1.06   0.22
Universal Basic Guys (FOX) 2024-10-27     0.68   0.26
        Fire Country (CBS) 2024-10-27     1.70   0.67
             Accused (FOX) 2024-10-22     0.95   0.37
  What Would You Do? (ABC) 2024-10-23     0.44   0.18


**Answer:** The biggest misses all come from small shows with ratings below one point, where a few tenths of a point turn into a very large percentage error. Krapopolis on FOX is the biggest, forecast 1.06 and actual 0.22. Its history mixes two Wednesday episodes at 1.76 and 1.93 with two Sunday episodes at 0.26 and 0.30, so the average is far above what a Sunday episode gets. Universal Basic Guys was forecast from two Sundays and got only 0.26 on Sunday 27 October, but on Tuesday 22 October after Accused it reached 1.85 against the same forecast of 0.68. Accused had one strong first episode at 1.87 and then 0.61 and 0.38, so its average of 0.95 was too high for the Tuesday that gave 0.37, while a Wednesday episode the next day reached 2.14. Fire Country on CBS was forecast from a Friday and a Saturday and then moved to a Sunday when the CBS schedule was pushed, 60 Minutes started at 19:35 and Fire Country was cut into two short pieces of 14 and 16 minutes, which gave 0.67 against 1.70. What Would You Do on ABC was forecast at 0.44 from only two earlier episodes and got 0.18 in the last hour of prime time on a Wednesday. In rating points the biggest miss was Dancing with the Stars on Sunday 27 October, 2.75 against 4.53. The heavy buyer forecast is a little worse, 38.6 percent against 35.8 percent, because the target group is much smaller. A telecast has about 348 panelists watching in the persons audience and only about 63 heavy buyers, so the sampling noise of a rating is about 13 percent for heavy buyers and about 5 percent for all persons, and the history average is noisy too. The gap is small because most of the error comes from the schedule and from programs moving between nights, and that hits both audiences in the same way. A better forecast would use the history of the same weekday and time slot instead of a plain average of all episodes. It would also adjust for the lead in and for schedule changes such as a pushed evening, and for heavy buyers it could forecast the persons rating and multiply it by a steady index.

### Task 5.1 - The plan

In [31]:
# cheapest target CPM first, at most 2 spots per telecast in a week
plan = {'Unsellable Houses (HGTV)': 6, 'Halloween Wars (FOOD)': 4, 'Grotesquerie (FX)': 11,
        'Hannity (FNC)': 10, 'Dancing with the Stars (ABC)': 2}

res_5_1 = pd.DataFrame({'spots': pd.Series(plan)})
res_5_1['cost'] = res_5_1['spots'] * res_2_1['cost per :30']
res_5_1['target impressions (M)'] = res_5_1['spots'] * res_2_1['target viewers (k)'] / 1000
res_5_1.index.name = 'program'

# table: program, spots, cost, target impressions
print(res_5_1.to_string(formatters={'cost': '{:.0f}'.format, 'target impressions (M)': '{:.3f}'.format}))
print("Plan cost ($):", round(res_5_1['cost'].sum()))
print("Plan target impressions (M):", round(res_5_1['target impressions (M)'].sum(), 1))

                              spots   cost target impressions (M)
program                                                          
Unsellable Houses (HGTV)          6  77400                  1.903
Halloween Wars (FOOD)             4  38800                  0.648
Grotesquerie (FX)                11  83512                  1.367
Hannity (FNC)                    10 399000                  4.169
Dancing with the Stars (ABC)      2 317200                  3.128
Plan cost ($): 915912
Plan target impressions (M): 11.2


### Task 5.2 Recommendation

**Answer:** I would buy 31 spots for about 916 thousand dollars, most of it on 10 spots of Hannity on Fox News and 2 spots of Dancing with the Stars on ABC, and fill the rest with cheap spots on Grotesquerie, Unsellable Houses and Halloween Wars. I chose by target CPM, because this plan gives about 11.2 million heavy buyer impressions at an average of about 82 dollars per thousand, while the football games and NCIS would cost 148 to 174 dollars per thousand and give roughly half the volume for the same money. I did not chase the highest index, since Unsellable Houses has the best index of 126.8 but only 317 thousand heavy buyers and room for about 6 spots a week, so it works as a cheap addition and not as the base of the plan. The plan is built on price and volume and not on concentration, because only about 18 percent of the impressions and of the money reach heavy buyers, a little under their 19.6 percent share among all adults, so about 82 cents of every dollar reach people who are not heavy buyers. The delivery is uncertain, because in Task 4 a single telecast missed its forecast by about 39 percent on average for heavy buyers, and although the errors partly cancel across many telecasts, the two Dancing with the Stars episodes carry more than a quarter of the impressions, so one schedule change there can move the total a lot. For that reason I would plan on a range and not on the single number 11.2 million, ask for a guarantee below it with make goods as in Lecture 6, and keep the unspent 84 thousand dollars as a reserve for extra spots if the first days under deliver. After the week I would measure the real heavy buyer impressions and the cost per thousand for each program, the unique reach and the frequency, because repeated spots on Hannity and Grotesquerie reach the same people many times, and I would check whether Kestrel sales rise among the exposed homes.

